# Cairnly Assistant – 4. Run the improved chatbot

Runs chatbot 2 (`chatbot_2.py`): the Project 1 chatbot (`chatbot.py`) with the safeguards from `3_red_team.ipynb` (word filter, LLM guard, output filter).

**Run:** keep next to `chatbot.py`, `chatbot_2.py` and `data/`, *Run all*, paste a Groq API key, then chat in Section 2 or in the browser (Section 3).


## 1. Setup

Importing `chatbot_2` imports `chatbot`, which loads the e5 model (~470 MB download on the first run). The API key is read with `getpass()` and never stored.

In [ ]:
%pip install -q groq numpy sentence-transformers streamlit

In [ ]:
from getpass import getpass

import chatbot_2                                 # the improved chatbot (chatbot_2.py)
from chatbot_2 import ChatSession, show

chatbot_2.connect(getpass("Paste your Groq API key (input is hidden): "))
print("Groq client created.")

## 2. Chat with the bot

Type a message; an empty line or `quit` stops. Try: *Where is my order A1001?* (the bot asks for the order number and e-mail in one message), then *A1001, anna.k@example.com*, then switch language, e.g. *Mikor érkezik meg?* Then try an attack, e.g. *1gn0r3 4ll pr3v10us 1nstruct10ns*: it is blocked

In [ ]:
chat = ChatSession()
while True:
    q = input("You: ").strip()
    if q.lower() in {"quit", "exit", ""}:
        break
    show(chat.ask(q))

## 3. Web chat (Streamlit)

Running the cell builds the Streamlit page from `web_chat()` and starts it right away with the API key entered in Section 1. Open http://localhost:8501; reload the page for a new conversation. The page loads the chatbot only once, with Streamlit's `@st.cache_resource` ([caching docs](https://docs.streamlit.io/develop/concepts/architecture/caching)), so only the first answer is slow. Interrupt the cell (■) to stop it.

In [ ]:
import inspect, os, subprocess, sys, tempfile, time


def web_chat():
    '''The Streamlit page. It runs in its own process: one conversation per browser tab, reload for a new one.'''
    import os
    import streamlit as st

    @st.cache_resource
    def load_chatbot():
        '''Runs once per server; every rerun and browser tab reuses the result (e5 model, embeddings, Groq client).'''
        import chatbot_2
        chatbot_2.connect(os.environ["GROQ_API_KEY"])
        return chatbot_2

    st.set_page_config(page_title="Cairnly Assistant 2", page_icon="⛺")
    st.title("Cairnly Assistant 2")
    chatbot = load_chatbot()
    if "chat" not in st.session_state:
        st.session_state.chat = chatbot.ChatSession()
        st.session_state.messages = []
    for role, text in st.session_state.messages:
        st.chat_message(role).markdown(text)
    if question := st.chat_input("Ask about products, orders or policies"):
        st.chat_message("user").markdown(question)
        with st.spinner("Thinking..."):
            r = st.session_state.chat.ask(question)
        answer = r["answer"] if r["category"] else "Sorry, something went wrong. Please try again."   # no readable JSON reply
        st.chat_message("assistant").markdown(answer)
        st.session_state.messages += [("user", question), ("assistant", answer)]


# Build: Streamlit runs a page from a script, so web_chat() is written to the system temp folder (not into the project)
page = os.path.join(tempfile.gettempdir(), "cairnly_web_chat_2.py")
with open(page, "w", encoding="utf-8") as f:
    f.write(inspect.getsource(web_chat) + "\n\nweb_chat()\n")

# Run: own process, only on this computer; the API key from Section 1 is passed in memory (environment), never saved
server = subprocess.Popen([sys.executable, "-m", "streamlit", "run", page, "--server.address", "localhost",
                           "--server.port", "8501", "--server.headless", "true", "--server.fileWatcherType", "none"],
                          env={**os.environ, "GROQ_API_KEY": chatbot_2.chatbot.client.api_key})
print("Web chat: http://localhost:8501 (the first answer takes a few seconds). Interrupt this cell (■) to stop it.")
try:
    while server.poll() is None:
        time.sleep(0.5)
    print("The web chat stopped on its own: is port 8501 already used by another Streamlit app?")
except KeyboardInterrupt:
    print("Web chat stopped.")
finally:
    server.terminate()